In [12]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))
import utils.tools as tools

In [13]:
# Suma kontrolna (hash) danych w pliku Parquet, z uwzględnieniem sortowania po kolumnach TowId i Data.

nazwa_pliku = "fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"../../dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet   c327308e20f97f828594a6666a7dd61d7e9f1307b6ce8f9a788ad07fdeebca60


In [14]:
df_kalendarz = pd.read_parquet("../../dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")


In [15]:


df_sprzedaz = df_kalendarz[df_kalendarz['TypDok'] == 21].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")


TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161


In [16]:
pelny_kalendarz.columns

Index(['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz',
       'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu',
       'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono',
       'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow',
       'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik',
       'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName',
       'JestMartwyHistorycznie'],
      dtype='str')

In [17]:

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort',
                  'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwyHistorycznie']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)


# Zakres życia per TowId (z bufor 14 dni,)
daty_sku = df_sprzedaz.groupby('TowId')['Data'].agg(DataStart='min', DataKoniec='max').reset_index()
daty_sku['DataKoniec'] = (daty_sku['DataKoniec'] + pd.Timedelta(days=14)).clip(upper=data_max)

# Dołączamy do pelny_kalendarz i filtrujemy
pelny_kalendarz = pelny_kalendarz.merge(daty_sku, on='TowId', how='left')

przed = len(pelny_kalendarz)
pelny_kalendarz = pelny_kalendarz[
    (pelny_kalendarz['Data'] >= pelny_kalendarz['DataStart']) &
    (pelny_kalendarz['Data'] <= pelny_kalendarz['DataKoniec'])
].copy()

print(f"Przed przycięciem: {przed:,} wierszy")
print(f"Po przycięciu:     {len(pelny_kalendarz):,} wierszy")

# Opcjonalnie: usuń pomocnicze kolumny DataStart/DataKoniec, jeśli nie chcesz ich w finalnym pliku
pelny_kalendarz = pelny_kalendarz.drop(columns=['DataStart', 'DataKoniec'])




print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")


Przed przycięciem: 15,810,161 wierszy
Po przycięciu:     8,802,515 wierszy

Puste rekordy (DokId=-1): 5,428,540
Realne rekordy sprzedaży: 3,373,975


In [18]:
pelny_kalendarz.sample(5)

,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,NazwaAsort,Dokument,WplywNaStan,MetodaLiczenia,Mnoznik,TypRuchu,CzyNiechciane,NazwaTowCleanName,NazwaAsortCleanName,JestMartwyHistorycznie
3466090,37777,2024-10-16,1820843,4.0,4.0,4.0,1.0,0.0,2.8476,2.8476,...,JOGURTY /nabiał,DF,True,IP,-1.0,sprzedaz,False,serek homo waniliowy 150g piatnica,jogurty nabial,False
935778,7407,2024-01-16,1537576,2.0,2.0,4.0,1.0,0.0,4.2900,4.2900,...,MĄKI /zbożowe i sypkie,DF,True,IP,-1.0,sprzedaz,False,maka szymanowska typ 480 1kg polskie mlyny,maki zbozowe i sypkie,False
1546254,12987,2024-11-15,1845980,7.0,7.0,4.0,1.0,0.0,2.4309,2.4309,...,1 PIWO BUTELKA,DF,True,IP,-1.0,sprzedaz,False,piwo but lomza jasne 500ml b zwr van pur,1 piwo butelka,False
5563517,12988,2025-02-16,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1 PIWO BUTELKA,NaN,NaN,NaN,NaN,NaN,NaN,piwo but lomza miodowe 500ml b zwr van pur,1 piwo butelka,False
6381642,5702,2025-02-17,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,DEKORACJE,NaN,NaN,NaN,NaN,NaN,NaN,polewa twarda mleczna 100g oetker,dekoracje,False


In [19]:
# Uzupełnianie brakujące wartości w kolumnach
pelny_kalendarz['Kolejnosc'] = pelny_kalendarz['Kolejnosc'].fillna(0)
pelny_kalendarz['NrPozycji'] = pelny_kalendarz['NrPozycji'].fillna(0)
pelny_kalendarz['TypPoz'] = pelny_kalendarz['TypPoz'].fillna(4)
pelny_kalendarz['IloscPlus'] = pelny_kalendarz['IloscPlus'].fillna(0)
pelny_kalendarz['IloscMinus'] = pelny_kalendarz['IloscMinus'].fillna(0)
pelny_kalendarz['CenaPoRab'] = pelny_kalendarz['CenaPoRab'].ffill()
pelny_kalendarz['Wartosc'] = pelny_kalendarz['Wartosc'].fillna(0)
pelny_kalendarz['KolejnyWDniu'] = pelny_kalendarz['KolejnyWDniu'].fillna(-1)
pelny_kalendarz['NrDok'] = pelny_kalendarz['NrDok'].fillna('brak')
pelny_kalendarz['TypDok'] = pelny_kalendarz['TypDok'].fillna(21)
pelny_kalendarz['AktywnyDok'] = pelny_kalendarz['AktywnyDok'].fillna(1)
pelny_kalendarz['Razem'] = pelny_kalendarz['Razem'].fillna(0)
pelny_kalendarz['DoZaplaty'] = pelny_kalendarz['DoZaplaty'].fillna(0)
pelny_kalendarz['Zaplacono'] = pelny_kalendarz['Zaplacono'].fillna(0)
pelny_kalendarz['Opis1'] = pelny_kalendarz['Opis1'].fillna('brak')
pelny_kalendarz['Producent'] = pelny_kalendarz['Producent'].fillna(-1)
pelny_kalendarz['AktywnyTow'] = pelny_kalendarz['AktywnyTow'].fillna(1)
pelny_kalendarz['Dokument'] = pelny_kalendarz['Dokument'].fillna("DF")
pelny_kalendarz['WplywNaStan'] = pelny_kalendarz['WplywNaStan'].fillna(False).astype(int)
pelny_kalendarz['MetodaLiczenia'] = pelny_kalendarz['MetodaLiczenia'].fillna('IP')
pelny_kalendarz['Mnoznik'] = pelny_kalendarz['Mnoznik'].fillna(0)
pelny_kalendarz['TypRuchu'] = pelny_kalendarz['TypRuchu'].fillna('sprzedaz')
pelny_kalendarz['CzyNiechciane'] = pelny_kalendarz['CzyNiechciane'].fillna(False).astype(int)
pelny_kalendarz['JestMartwyHistorycznie'] = pelny_kalendarz['JestMartwyHistorycznie'].astype(int)

In [20]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


Wiersze w df_sprzedaz (surowe): 3,373,975
Realne rekordy w pelny_kalendarz: 3,373,975
Zgodność: True

Rozmiar: (8802515, 33)
Pamięć: 3395.0 MB

Kolumny: ['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow', 'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwyHistorycznie']


In [21]:
pelny_kalendarz.columns

Index(['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz',
       'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu',
       'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono',
       'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow',
       'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik',
       'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName',
       'JestMartwyHistorycznie'],
      dtype='str')

In [22]:
pelny_kalendarz.sample(5)

,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,NazwaAsort,Dokument,WplywNaStan,MetodaLiczenia,Mnoznik,TypRuchu,CzyNiechciane,NazwaTowCleanName,NazwaAsortCleanName,JestMartwyHistorycznie
3601567,49541,2023-11-21,-1,0.0,0.0,4.0,0.0,0.0,1.8900,0.0000,...,***PRZECENY,DF,0,IP,0.0,sprzedaz,0,przec deser satino o sm karmelowym z bita smie...,przeceny,0
7707442,51603,2025-06-25,-1,0.0,0.0,4.0,0.0,0.0,11.3740,0.0000,...,3 WÓDKI I NAP.ALK POWYŻEJ 18%,DF,0,IP,0.0,sprzedaz,0,napoj alk barmanska pigwa 21 0 5l janton,3 wodki i nap alk powyzej 18,0
3094801,5257,2023-04-05,-1,0.0,0.0,4.0,0.0,0.0,4.1574,0.0000,...,HERBATY,DF,0,IP,0.0,sprzedaz,0,herb loyd grzaniec malinowy 10tb mokate,herbaty,0
8691455,53628,2024-07-09,-1,0.0,0.0,4.0,0.0,0.0,2.4286,0.0000,...,***PRZECENY,DF,0,IP,0.0,sprzedaz,0,przec jogurt fantasia stars 102g danon,przeceny,0
3997934,52900,2025-06-09,2030240,8.0,8.0,4.0,1.0,0.0,2.8374,2.8374,...,MARKA WŁASNA SPAR,DF,1,IP,-1.0,sprzedaz,0,zelki babeczki 90g spar,marka wlasna spar,0


In [23]:
pelny_kalendarz.to_parquet(
    "../../dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [24]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"../../dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   kalendarz_pelny_towid.parquet   defac931291e6bad894c972b985ab043503fa4bb1ee673152bdcc24d27aa7c17


In [25]:
# hash pliku kalendarz_pelny_towid.parquet:  defac931291e6bad894c972b985ab043503fa4bb1ee673152bdcc24d27aa7c17